# Scalar *vs* Array Evaluation of Splines
The tables give the number of times the array evaluation of splines is faster than the repeated evaluations at scalar arguments.

## Caveat
The results provide representative timings only when the notebook is run natively with a properly installed ``splinekit`` library.

In [ ]:
# Load the required libraries
import math
import numpy as np
import time

import splinekit as sk # This library

# Experimental conditions
highest_degree = 9 # Number or tables
periods = [2, 5, 10, 20, 50, 100, 200, 500, 1000] # Period of the spline
highest_oversampling = 6 # Number of samples per unit length
repeats = 10 # Number of times one experiment is performed

for degree in range(1, highest_degree + 1):
    # Table of results
    print()
    print("Acceleration for Spline Degree n =", degree)
    print("====================================================================")
    print("      Period |     2     5    10    20    50   100   200   500  1000")
    print("Oversampling |")
    print("-------------+------------------------------------------------------")
    for oversampling in range(1, highest_oversampling + 1):
        performance = [0.0]
        for period in periods:
            # Create a random spline
            f = sk.PeriodicSpline1D.from_spline_coeff(
                np.random.standard_normal(period), # Fresh data
                degree = degree,
                delay = np.random.standard_normal() # Random delay
            )
            # Random starting points
            starting_points = np.random.standard_normal(repeats)

            # Duration of "get_samples"
            start = time.perf_counter()
            for x0 in starting_points:
                f.get_samples(x0, support_length = period, oversampling = oversampling)
            end = time.perf_counter()
            get_samples_duration = end - start

            # Duration of repeated calls to "at"
            start = time.perf_counter()
            for x0 in starting_points:
                for q in range(period * oversampling):
                    f.at(x0 + q / oversampling)
            end = time.perf_counter()
            at_duration = end - start
            
            # Relative speed
            performance += [
                at_duration / get_samples_duration if 0 != get_samples_duration else float("nan")
            ]
        print("M = {0:1d}        | {1:5.2f} {2:5.2f} {3:5.2f} {4:5.2f} {5:5.2f} {6:5.2f} {7:5.2f} {8:5.2f} {9:5.2f}".format(
            oversampling,
            performance[1],
            performance[2],
            performance[3],
            performance[4],
            performance[5],
            performance[6],
            performance[7],
            performance[8],
            performance[9]
        ))
    print("====================================================================")


###################
#                 #
#   Be patient!   #
#                 #
###################


# Duration of the computations on a desktop computer of year 2021: ~15s

